In [23]:
import torch
import torch.nn as nn
import math

In [24]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
    def forward(self, x):
        return self.embedding(x)

In [25]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_seq_length, d_model):
        super().__init__()
        pe = torch.zeros(max_seq_length, d_model)
        positions = torch.arange(0, max_seq_length).unsqueeze(-1)
        div_term = torch.exp(-2 * torch.arange(0, d_model, 2) /d_model * math.log(10000) )  
        pe[:, 0::2] =  torch.sin(positions * div_term)
        pe[:, 1::2] = torch.cos(positions * div_term)
        pe.unsqueeze_(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return (x + self.pe[:, :x.shape[1], :])

In [26]:
class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        assert d_model % n_heads == 0        
        self.d_k = d_model//n_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)


    @staticmethod
    def calc_attention(query, key, value, mask, d_k):
        matrix = torch.matmul(query, key.transpose(-2, -1))
        if mask is not None:
              matrix = matrix.masked_fill(mask ==0, 1e-9)
        attention_scores = torch.matmul(torch.softmax(matrix/ math.sqrt(d_k), dim = -1),value)

        return attention_scores


    def forward(self, q, k, v, mask):
        query = self.w_q(q).reshape(q.shape[0], q.shape[1], self.n_heads, self.d_k).transpose(2,1)
        key = self.w_k(k).reshape(k.shape[0], k.shape[1], self.n_heads, self.d_k).transpose(2,1)
        value = self.w_v(v).reshape(v.shape[0], v.shape[1], self.n_heads, self.d_k).transpose(2, 1)
        out_matrix = MultiHeadAttentionBlock.calc_attention(query, key, value, mask, self.d_k)
        out_matrix = out_matrix.transpose(1, 2).contiguous()
        out_matrix = out_matrix.reshape(
            q.shape[0],
            q.shape[1],
            self.d_model
        )

        return self.w_o(out_matrix)

In [27]:
class FeedForwardBlock(nn.Module):
    def __init__(self, d_model, dff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, dff)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(dff, d_model)

    def forward(self ,x):
        return self.linear2(self.relu(self.linear1(x)))

In [28]:
class LayerNormalization(nn.Module):
    def __init__(self, d_model,eps =10**-6):
        super().__init__()
        self.eps  = eps
        self.alpha = nn.Parameter(torch.ones(d_model))
        self.gamma = nn.Parameter(torch.zeros(d_model))

    
    def forward(self, x):
        mean = x.mean(dim = -1, keepdim = True)
        std = x.std(dim = -1, keepdim = True)

        return self.alpha *(x - mean /(std + self.eps)) + self.gamma

In [29]:
class ResidualConnection(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = LayerNormalization(d_model)
    
    def forward(self, x, sublayer:nn.ModuleList):
        return x + sublayer(self.norm(x))

In [30]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.residual_connection = nn.ModuleList([ResidualConnection(d_model) for _ in range(2)])

    def forward(self, x, src_mask, multi_head_attention : MultiHeadAttentionBlock, feed_forward: FeedForwardBlock):
        x = self.residual_connection[0](x, lambda x: multi_head_attention(x, x, x, src_mask))
        x = self.residual_connection[1](x, feed_forward)
        return x


In [31]:
class Encoder(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.layers = layers

    def forward(self, x,  src_mask, multi_head_attention : MultiHeadAttentionBlock, feed_forward : FeedForwardBlock):
        for layer in self.layers:
            x = layer(x, src_mask, multi_head_attention, feed_forward)

        return x


In [32]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.residual_connections = nn.ModuleList([ResidualConnection(d_model) for _  in range(3)])

    def forward(self, x, encoder_output, src_mask, tgt_mask, multi_head_attention: MultiHeadAttentionBlock, feed_forward : FeedForwardBlock):
        x = self.residual_connections[0](x, lambda x: multi_head_attention(x, x, x, tgt_mask))
        x = self.residual_connections[1](x, lambda x : multi_head_attention(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connections[2](x, feed_forward)

        return x

In [33]:
class Decoder(nn.Module):
    def __init__(self, layers):
        super().__init__()
        self.layers = layers

    def forward(self, x, encoder_output, src_mask, tgt_mask, multi_head_attention, feed_forward):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask, multi_head_attention, feed_forward)

        return x

In [34]:
class ProjectionLayer(nn.Module):
    def __init__(self, d_model, tgt_vocab_size):
        super().__init__()
        self.layer = nn.Linear(d_model, tgt_vocab_size)


    def forward(self, x):
        return self.layer(x)


In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, src_max_seq_len, tgt_max_seq_len,  d_model = 512, dff =2048, n_heads = 8, n_blocks =6):
        super().__init__()
        self.src_embed = Embedding(src_vocab_size, d_model)
        self.src_position_encoding  = PositionalEncoding(src_max_seq_len, d_model)
        self.tgt_embed = Embedding(tgt_vocab_size, d_model)
        self.tgt_position_encoding = PositionalEncoding(tgt_max_seq_len, d_model)
        self.attention_block = MultiHeadAttentionBlock(d_model, n_heads)
        self.feed_forward = FeedForwardBlock(d_model, dff)
        self.projection  = ProjectionLayer(d_model, tgt_vocab_size)
        encoder_blocks = []
        decoder_blocks = []
        for _ in range(n_blocks):
            encoder_layer  = EncoderLayer(d_model)
            encoder_blocks.append(encoder_layer)

        self.encoder = Encoder(nn.ModuleList(encoder_blocks))

        for _ in range(n_blocks):
            decoder_layer = DecoderBlock(d_model)
            decoder_blocks.append(decoder_layer)

        self.decoder = Decoder(nn.ModuleList(decoder_blocks))


    def encode(self, x, src_mask):
        x = self.src_embed(x)
        x = self.src_position_encoding(x)
        out = self.encoder(x, src_mask, self.attention_block, self.feed_forward)
        return out


    def decode(self, x, encoder_output, src_mask, tgt_mask):
        x  = self.tgt_embed(x)
        x = self.tgt_position_encoding(x)
        out = self.decoder(x, encoder_output, src_mask, tgt_mask, self.attention_block, self.feed_forward)
        return out


    def project(self, x):
        return self.projection(x)

